# creditport Demo

Credit index options pricing, portfolio greeks, scenario analysis, and Monte Carlo simulation.

In [ ]:
import datetime as dt
from creditport import (
    Portfolio, MarketData, IndexMarketData, DiscountCurve,
    ScenarioEngine, MonteCarlo, MonteCarloConfig,
    IndexFamily, OptionType,
    compute_delta_hedge, apply_delta_hedge,
)

## 1. Load Market Data

You can load from CSV/Excel or build manually.

In [ ]:
# Option A: Load from CSV
market = MarketData.from_csv(
    "../data/examples/sample_market.csv",
    ref_date=dt.date(2026, 2, 7),
    risk_free_rate=0.03,
)

# Check what's loaded
for key, idx in market.indices.items():
    print(f"{idx.family.value}{idx.series}: spread={idx.spread_bps}bp, "
          f"vol surface: {len(idx.vol_surface)} strikes")

In [ ]:
# Option B: Build manually
market_manual = MarketData(
    ref_date=dt.date(2026, 2, 7),
    discount_curve=DiscountCurve(rate=0.03),
)
market_manual.add_index(IndexMarketData(
    family=IndexFamily.ITRAXX_MAIN,
    series=44,
    spread_bps=60,
    vol_surface={50: 0.47, 55: 0.43, 60: 0.40, 65: 0.38, 70: 0.36},
))

## 2. Build a Portfolio

Position notation:
- `main44 mar55p` = iTraxx Main S44, March expiry, 55bp strike, payer
- `cdxig43 jun80r` = CDX IG S43, June expiry, 80bp strike, receiver
- `main44 5y` = Linear index position (for delta hedging)

In [ ]:
port = Portfolio(ref_date=dt.date(2026, 2, 7))

# Long a payer spread: buy 55 payer, sell 70 payer
port.add("main44 jun55p", notional=10_000_000)
port.add("main44 jun70p", notional=-10_000_000)

# Also long a receiver
port.add("main44 jun50r", notional=5_000_000)

print(f"Portfolio has {len(port.positions)} positions")

## 3. Price and Greeks

In [ ]:
# Full greeks table
greeks_df = port.greeks_table(market)
greeks_df

In [ ]:
# Formatted summary
print(port.summary(market))

## 4. Delta Hedge

Compute and optionally apply a delta hedge using the underlying index.

In [ ]:
# Compute hedge (doesn't modify portfolio)
hedge = compute_delta_hedge(port, market, hedge_index="main44 5y")
print(f"Delta before hedge: {hedge.portfolio_delta_before:,.0f}")
print(f"Hedge notional:     {hedge.hedge_notional:,.0f}")
print(f"Delta after hedge:  {hedge.portfolio_delta_after:,.4f}")

In [ ]:
# Apply hedge (adds position to portfolio)
hedge = apply_delta_hedge(port, market, "main44 5y")
print("\nPortfolio with hedge:")
print(port.summary(market))

## 5. Scenario Analysis

Parallel spread shifts, vol shifts, time decay, and 2D matrices.

In [ ]:
engine = ScenarioEngine(market, port)

# Parallel spread shift
results = engine.parallel_shift(spread_shifts=[-20, -10, -5, 0, 5, 10, 20])
engine.results_table(results)

In [ ]:
# 2D spread x vol matrix
matrix = engine.spread_vol_matrix(
    spread_shifts=[-20, -10, -5, 0, 5, 10, 20],
    vol_shifts=[-0.10, -0.05, 0, 0.05, 0.10],
)
matrix.style.format("{:,.0f}").background_gradient(cmap="RdYlGn", axis=None)

In [ ]:
# Time decay
decay = engine.time_decay(days=[0, 1, 5, 10, 20, 30, 60, 90])
decay_df = engine.results_table(decay)
decay_df

In [ ]:
import matplotlib.pyplot as plt

# Plot P&L vs spread shift
results_df = engine.results_table(results)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(results_df["spread_shift"], results_df["pnl"], "o-")
ax1.axhline(0, color="gray", linestyle="--", alpha=0.5)
ax1.set_xlabel("Spread Shift (bps)")
ax1.set_ylabel("P&L")
ax1.set_title("P&L vs Spread Shift")
ax1.grid(True, alpha=0.3)

ax2.plot(decay_df["time_shift"], decay_df["pnl"], "o-", color="red")
ax2.axhline(0, color="gray", linestyle="--", alpha=0.5)
ax2.set_xlabel("Days Forward")
ax2.set_ylabel("P&L (Time Decay)")
ax2.set_title("Theta Profile")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Monte Carlo Simulation

Simulate spread paths and compute the P&L distribution over a horizon.

In [ ]:
mc = MonteCarlo(
    market=market,
    portfolio=port,
    config=MonteCarloConfig(
        n_paths=5000,
        horizon_days=90,
        seed=42,
    ),
)
mc_result = mc.run()
mc_result.summary_df()

In [ ]:
# P&L distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

dist = mc_result.pnl_distribution(bins=80)
ax1.bar(dist["pnl"], dist["freq"], width=(dist["pnl"].iloc[1] - dist["pnl"].iloc[0]) * 0.9)
ax1.axvline(mc_result.mean_pnl, color="red", linestyle="--", label=f"Mean: {mc_result.mean_pnl:,.0f}")
ax1.axvline(mc_result.var_95, color="orange", linestyle="--", label=f"95% VaR: {mc_result.var_95:,.0f}")
ax1.set_xlabel("P&L")
ax1.set_ylabel("Frequency")
ax1.set_title("90-Day P&L Distribution")
ax1.legend()

# Sample spread paths
key = list(mc_result.spread_paths.keys())[0]
paths = mc_result.spread_paths[key]
for i in range(min(100, len(paths))):
    ax2.plot(mc_result.time_grid, paths[i], alpha=0.1, color="steelblue")
ax2.set_xlabel("Days")
ax2.set_ylabel("Spread (bps)")
ax2.set_title(f"Simulated Spread Paths ({key[0].value}{key[1]})")

plt.tight_layout()
plt.show()